# GEO-Bench-VLM — Score Results

Load all result JSON files saved to Google Drive by the evaluation notebooks,
then compute per-task and overall accuracy for every model and print a comparison table.

**No GPU needed.** Run after at least one eval notebook has finished.

In [ ]:
# ── CONFIG — only edit this cell ─────────────────────────────────────────────
DRIVE_BASE  = '/content/drive/MyDrive/GEOBench'  # must match your eval notebooks
REPO_URL    = 'https://github.com/Tarekbouamer/GEOBench-VLM'
RESULTS_DIR = f'{DRIVE_BASE}/results'             # must match RESULTS_DIR in eval notebooks


In [ ]:
# ── 1. Mount Drive ────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

os.makedirs(RESULTS_DIR, exist_ok=True)

model_dirs = [d for d in os.listdir(RESULTS_DIR)
              if os.path.isdir(os.path.join(RESULTS_DIR, d))] \
             if os.path.isdir(RESULTS_DIR) else []

print(f'RESULTS_DIR = {RESULTS_DIR}')
print(f'Model result folders found: {model_dirs if model_dirs else "(none yet — run an eval notebook first)"}')


In [ ]:
# ── 2. Clone repo (for score_results.py) ─────────────────────────────────────
import os
REPO = '/content/GEO-Bench-VLM'
if not os.path.isdir(REPO):
    !git clone {REPO_URL} {REPO}
%cd {REPO}
print('Repo ready.')

In [ ]:
# ── 3. Run scorer ─────────────────────────────────────────────────────────────
import subprocess, sys

result = subprocess.run(
    [sys.executable, 'score_results.py',
     '--results_dir', RESULTS_DIR,
     '--output_dir',  RESULTS_DIR],
    cwd=f'{REPO}/eval_geobenchvlm',
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)


In [ ]:
# ── 4. Display scores JSON ────────────────────────────────────────────────────
import json, os

scores_path = os.path.join(RESULTS_DIR, 'scores.json')
if not os.path.exists(scores_path):
    print('scores.json not found — did the scorer run successfully?')
else:
    with open(scores_path) as f:
        scores = json.load(f)
    print(json.dumps(scores, indent=2))


In [ ]:
# ── 5. Display scores as a DataFrame table ────────────────────────────────────
import json, os, pandas as pd

scores_path = os.path.join(RESULTS_DIR, 'scores.json')
if not os.path.exists(scores_path):
    print('scores.json not found.')
else:
    with open(scores_path) as f:
        scores = json.load(f)

    rows = []
    for model, data in scores.items():
        row = {'Model': model, **data.get('per_task', {}), 'OVERALL': data.get('overall')}
        rows.append(row)

    df = pd.DataFrame(rows).set_index('Model')
    df = df.applymap(lambda x: f'{x:.1f}%' if isinstance(x, (int, float)) else x)
    display(df)


In [ ]:
# ── 6. Display scores CSV ─────────────────────────────────────────────────────
import os, pandas as pd

csv_path = os.path.join(RESULTS_DIR, 'scores.csv')
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path, index_col=0)
    display(df)
    print(f'CSV saved at: {csv_path}')
else:
    print('scores.csv not found.')
